In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.neuron_simulation.config import NeuronSimulationConfig
from codes.network_params.loader import load_network_parameters
from codes.plotting import fig_plots

import codes.stimuli as stim

from codes.mf_simulation.config import MeanFieldSimulationConfig

from pydantic import BaseModel

project_path = repo_path / "projects" / "00_drafts_tests"
os.chdir(project_path)

class TestNeuronSimulationParams(BaseModel):
    neuron_simulation: NeuronSimulationConfig

class TestMFSimulationParams(BaseModel):
    mf_simulation: MeanFieldSimulationConfig


In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
simulators = ["zerlaut2018", "pynn.nest"]


In [ ]:
from codes.stimuli.loader import load_stimuli_config
stimuli_config = load_stimuli_config(project_path / "params" / "stimuli.yaml")

In [ ]:
test_workflow_params = {}
test_workflow_params["mf_simulation"] = {
    "execution_mode": "run",  # run, load
    "simulator": "tvb",
    "model": "divolo2019.second_order",
    "time_step": 0.1,
    "resolution_time": 40.0,  # T
    "seed": 42,
    "init_values": {
        "E": [0.0, 0.0],
        "I": [0.0, 0.0],
        "C_ee": [0.5, 0.5],
        "C_ei": [0.5, 0.5],
        "C_ii": [0.5, 0.5],
        "W_e": [0.05, 0.05],
        "W_i": [0.0, 0.0],
        # "X_e": [1.0, 1.0],
        # "Y_e": [0.0, 0.0],
        # "X_i": [1.0, 1.0],
        # "Y_i": [0.0, 0.0],
        "noise": [0.0, 0.0],
        "stimulus": [0.0, 0.0],
    },
    "transfer_function": {
        "fit_transfer_function" : True,
        "tf_model": {
            "model_name": "neuropsi.custom",
            "square_terms": True,
            "log_term": False,
            "adaptation": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            },
        },
        "TF_fitting" : {
            "method" : "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp" : False,
                "maxiter" : 10000
            }
        },
        "tf_fits": {
            "exc_neuron" : {
                # [-49.8, 5.06, -25.0, 1.4, -0.41, 10.5, -36.0, 7.4, 1.2, -40.7]      # [mV], coefficients of the polynomial
                "P_0" : -49.8,
                "P_mean" : 5.06,
                "P_std": -25.0,
                "P_tau" : 1.4,
                "P_log": 0.0,
                "P_mean_mean" : -0.41,
                "P_mean_std" : 7.4,
                "P_mean_tau" : 1.2,
                "P_std_std" : 10.5,
                "P_std_tau" : -40.7,
                "P_tau_tau" : -36.0,
            },
            "inh_neuron" : {
                # [-51.4, 4.0, -8.3, 0.2, -0.5, 1.4, -14.6, 4.5, 2.8, -15.3]          # [mV], coefficients of the polynomial
                "P_0" : -51.4,
                "P_mean" : 4.0,
                "P_std": -8.3,
                "P_tau" : 0.2,
                "P_log": 0.0,
                "P_mean_mean" : -0.5,
                "P_mean_std" : 4.5,
                "P_mean_tau" : 2.8,
                "P_std_std" : 1.4,
                "P_std_tau" : -15.3,
                "P_tau_tau" : -14.6,
            },
        }
    }
}

mf_sim_params = TestMFSimulationParams(**test_workflow_params).mf_simulation

In [ ]:
from codes.mf_simulation import run_mf_simulation_workflow

mf_results = run_mf_simulation_workflow(
        mf_sim_params=mf_sim_params, 
        network_params=network_params,
        stimuli_dict={"Sinusoidal": stimuli_config["Sinusoidal"]}
    )

In [ ]:
mf_results["Sinusoidal"]["data"].shape

In [ ]:
stimuli_config["TwoSidedGaussian"].stim_params

In [ ]:
test_workflow_params